In [ ]:
# ============================================================
# EXFOR data collector for Cr-50 / Cr-53 -> CSV -> ZIP
# Run this in Google Colab, one cell at a time (or all together)
# ============================================================

# --- Cell 1: install package ---
!pip install x4i3 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.1/270.1 kB 5.6 MB/s eta 0:00:00


In [ ]:

# --- Cell 2: imports ---
import os
import shutil
import zipfile
import pandas as pd
from x4i3 import exfor_manager

Using database version x4i3_X4-2023-04-29 located in: /usr/local/lib/python3.13/dist-packages/x4i3/data


236MB [00:03, 76.9MB/s]                         


Decompressing archive x4i3_X4-2023-04-29.tar.gz


100%|██████████| 25796/25796 [00:17<00:00, 1492.58it/s]


Installed database version x4i3_X4-2023-04-29


In [ ]:

# --- Cell 4: define what to search for ---
# targets: EXFOR internal naming is usually "24-CR-50" (Z-symbol-A)
targets = {
    "Cr-50": "24-CR-50",
    "Cr-53": "24-CR-53",
}

# reactions of interest, in EXFOR notation
reactions = {
    "n_g": "N,G",   # (n,gamma) capture -- required
    "n_a": "N,A",   # (n,alpha) -- for fusion damage section
    "n_p": "N,P",   # (n,proton) -- optional
}

output_dir = "exfor_output"
os.makedirs(output_dir, exist_ok=True)


In [ ]:

# --- Cell 5: run the queries and save each result as CSV ---
summary_log = []

for iso_label, target_code in targets.items():
    for rxn_label, rxn_code in reactions.items():
        print(f"Searching: target={target_code}, reaction={rxn_code} ...")
        try:
            results = db.retrieve(target=target_code, reaction=rxn_code)
        except Exception as e:
            print(f"  -> query failed: {e}")
            summary_log.append([iso_label, rxn_label, "QUERY_ERROR", str(e)])
            continue

        if not results or len(results) == 0:
            print(f"  -> no entries found")
            summary_log.append([iso_label, rxn_label, "NO_DATA", ""])
            continue

        entry_count = 0
        for entry_id, entry in results.items():
            try:
                # each entry may contain multiple datasets (subentries)
                for ds_id, dataset in entry.datasets.items():
                    try:
                        df = dataset.getSimplified()  # returns pandas-friendly data
                        if df is None or len(df) == 0:
                            continue
                        fname = f"{output_dir}/EXFOR_{iso_label}_{rxn_label}_{entry_id}_{ds_id}.csv"
                        pd.DataFrame(df).to_csv(fname, index=False)
                        entry_count += 1
                    except Exception as e_ds:
                        print(f"    dataset {ds_id} skipped: {e_ds}")
            except Exception as e_entry:
                print(f"  entry {entry_id} skipped: {e_entry}")

        print(f"  -> saved {entry_count} dataset file(s)")
        summary_log.append([iso_label, rxn_label, "OK", f"{entry_count} datasets"])

Searching: target=24-CR-50, reaction=N,G ...
  -> query failed: name 'db' is not defined
Searching: target=24-CR-50, reaction=N,A ...
  -> query failed: name 'db' is not defined
Searching: target=24-CR-50, reaction=N,P ...
  -> query failed: name 'db' is not defined
Searching: target=24-CR-53, reaction=N,G ...
  -> query failed: name 'db' is not defined
Searching: target=24-CR-53, reaction=N,A ...
  -> query failed: name 'db' is not defined
Searching: target=24-CR-53, reaction=N,P ...
  -> query failed: name 'db' is not defined


In [ ]:
# --- Cell 6: write a summary log CSV ---
summary_df = pd.DataFrame(summary_log, columns=["isotope", "reaction", "status", "detail"])
summary_df.to_csv(f"{output_dir}/search_summary.csv", index=False)
print("\nSummary of all searches:")
print(summary_df)



Summary of all searches:
  isotope reaction       status                    detail
0   Cr-50      n_g  QUERY_ERROR  name 'db' is not defined
1   Cr-50      n_a  QUERY_ERROR  name 'db' is not defined
2   Cr-50      n_p  QUERY_ERROR  name 'db' is not defined
3   Cr-53      n_g  QUERY_ERROR  name 'db' is not defined
4   Cr-53      n_a  QUERY_ERROR  name 'db' is not defined
5   Cr-53      n_p  QUERY_ERROR  name 'db' is not defined


In [ ]:

# --- Cell 7: zip everything ---
zip_name = "exfor_cr_data.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(output_dir):
        for file in files:
            filepath = os.path.join(root, file)
            arcname = os.path.relpath(filepath, output_dir)
            zf.write(filepath, arcname)

print(f"\nDone. Download '{zip_name}' from the Colab file browser (left sidebar).")


Done. Download 'exfor_cr_data.zip' from the Colab file browser (left sidebar).


In [ ]:
# --- Cell 8 (Colab only): trigger download ---
try:
    from google.colab import files
    files.download(zip_name)
except ImportError:
    print("Not running in Colab -- zip file is saved locally, download manually.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>